In [36]:
import os
import re
import json
import random
import hashlib
import platform
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image, ImageFile
from tqdm.auto import tqdm

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 7
random.seed(SEED)
np.random.seed(SEED)

INPUT_ROOT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working/main19_manifest_audit")

OUTPUTS = {
    "root": WORK_ROOT,
    "tables": WORK_ROOT / "tables",
    "figures": WORK_ROOT / "figures",
    "reports": WORK_ROOT / "reports",
    "samples": WORK_ROOT / "samples",
}

for path in OUTPUTS.values():
    path.mkdir(parents=True, exist_ok=True)

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp", ".tif", ".tiff"
}

EXPECTED_FINAL_COUNTS = {
    "train": 18113,
    "calibration": 3194,
    "internal_test": 5322,
}

print("Python:", platform.python_version())
print("Pandas:", pd.__version__)
print("Input root:", INPUT_ROOT)
print("Work root:", WORK_ROOT)

Python: 3.13.15
Pandas: 2.3.3
Input root: /kaggle/input
Work root: /kaggle/working/main19_manifest_audit


In [37]:
def utc_now():
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")

def canonical_text(value):
    value = str(value).strip().lower()
    value = re.sub(r"[\s\-/]+", "_", value)
    value = re.sub(r"_+", "_", value)
    return value.strip("_")

def normalise_split(value):
    value = canonical_text(value)

    if value in {"train", "training"}:
        return "train"

    if "calib" in value:
        return "calibration"

    if value in {
        "test", "internal_test", "internaltest",
        "validation", "valid", "val"
    }:
        return "internal_test"

    return value

def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()

    with open(path, "rb") as file:
        while True:
            block = file.read(chunk_size)
            if not block:
                break
            digest.update(block)

    return digest.hexdigest()

def safe_read_csv(path, nrows=None):
    try:
        return pd.read_csv(
            path,
            low_memory=False,
            nrows=nrows,
        )
    except UnicodeDecodeError:
        return pd.read_csv(
            path,
            low_memory=False,
            nrows=nrows,
            encoding="latin1",
        )

def display_dataframe(df, rows=10):
    print("Shape:", df.shape)
    display(df.head(rows))

In [38]:
def get_dataset_roots(input_root):
    roots = []

    for path in sorted(input_root.iterdir()):
        if not path.is_dir():
            continue

        # Kaggle sometimes nests datasets as /kaggle/input/datasets/<owner>/<dataset>
        if path.name == "datasets":
            for owner_dir in path.iterdir():
                if owner_dir.is_dir():
                    for dataset_dir in owner_dir.iterdir():
                        if dataset_dir.is_dir():
                            roots.append(dataset_dir)
        else:
            roots.append(path)

    return roots

DATASET_ROOTS = get_dataset_roots(INPUT_ROOT)

print("Detected dataset roots:")
for dataset_root in DATASET_ROOTS:
    print("-", dataset_root)

Detected dataset roots:
- /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles
- /kaggle/input/datasets/mohitsingh1804/plantvillage


In [39]:
def directory_summary(root, max_depth=3):
    rows = []

    root_parts = len(root.parts)

    for path in root.rglob("*"):
        depth = len(path.parts) - root_parts

        if depth > max_depth:
            continue

        rows.append({
            "relative_path": str(path.relative_to(root)),
            "type": "directory" if path.is_dir() else "file",
            "depth": depth,
            "size_bytes": path.stat().st_size if path.is_file() else np.nan,
        })

    return pd.DataFrame(rows)

all_directory_summaries = []

for dataset_root in DATASET_ROOTS:
    summary = directory_summary(dataset_root, max_depth=3)
    summary["dataset_root"] = str(dataset_root)
    all_directory_summaries.append(summary)

directory_overview = pd.concat(all_directory_summaries, ignore_index=True)

directory_overview.to_csv(
    OUTPUTS["tables"] / "dataset_directory_overview.csv",
    index=False,
)

display_dataframe(directory_overview, rows=50)

Shape: (123, 5)


,relative_path,type,depth,size_bytes,dataset_root
0,plantvillage_selected_manifest.csv,file,1,15437583.0,/kaggle/input/datasets/katakuricharlotte/crop-...
1,split_outputs,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
2,nb_4567_8,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
3,nb_06,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
4,nb_05,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
5,plantvillage_selected_manifest_clean.csv,file,1,15678867.0,/kaggle/input/datasets/katakuricharlotte/crop-...
6,Eda-tables,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
7,best_efficientnetv2s,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
8,nb0_3,directory,1,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...
9,split_outputs/split_outputs,directory,2,NaN,/kaggle/input/datasets/katakuricharlotte/crop-...


In [40]:
def find_root_by_keywords(roots, keywords):
    keywords = [keyword.lower() for keyword in keywords]

    for root in roots:
        root_name = root.name.lower()

        if all(keyword in root_name for keyword in keywords):
            return root

    return None

PLANTVILLAGE_ROOT = find_root_by_keywords(
    DATASET_ROOTS,
    ["plantvillage"],
)

MANIFEST_ROOT = find_root_by_keywords(
    DATASET_ROOTS,
    ["manifest"],
)

print("Automatically detected PlantVillage root:", PLANTVILLAGE_ROOT)
print("Automatically detected manifest root:", MANIFEST_ROOT)

# If automatic detection is wrong, manually overwrite only these two lines.
# PLANTVILLAGE_ROOT = Path("/kaggle/input/plantvillage/PlantVillage")
# MANIFEST_ROOT = Path("/kaggle/input/crop-manifestfiles")

assert PLANTVILLAGE_ROOT is not None, (
    "Could not detect PlantVillage dataset root. "
    "Set PLANTVILLAGE_ROOT manually."
)

assert MANIFEST_ROOT is not None, (
    "Could not detect Crop-Manifestfiles dataset root. "
    "Set MANIFEST_ROOT manually."
)

Automatically detected PlantVillage root: /kaggle/input/datasets/mohitsingh1804/plantvillage
Automatically detected manifest root: /kaggle/input/datasets/katakuricharlotte/crop-manifestfiles


In [41]:
csv_rows = []

for csv_path in sorted(MANIFEST_ROOT.rglob("*.csv")):
    csv_rows.append({
        "csv_path": str(csv_path),
        "relative_path": str(csv_path.relative_to(MANIFEST_ROOT)),
        "size_mb": round(csv_path.stat().st_size / (1024 ** 2), 3),
    })

csv_inventory = pd.DataFrame(csv_rows)

assert len(csv_inventory) > 0, (
    f"No CSV files found under {MANIFEST_ROOT}"
)

csv_inventory.to_csv(
    OUTPUTS["tables"] / "manifest_csv_inventory.csv",
    index=False,
)

display(csv_inventory)

,csv_path,relative_path,size_mb
0,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/class_distribution_summa...,0.002
1,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/crop_distribution_summar...,0.000
2,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/cross_split_exact_phash_...,0.001
3,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/dataset_overview.csv,0.000
4,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/eda_findings_and_modelli...,0.002
5,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/excluded_exact_train_val...,0.002
6,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/healthy_diseased_summary...,0.000
7,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/image_metadata_summary.csv,0.000
8,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/near_duplicate_candidate...,0.058
9,/kaggle/input/datasets/katakuricharlotte/crop-...,Eda-tables/Eda-tables/plantvillage_selected_ma...,14.950


In [42]:
schema_rows = []

for csv_path_str in tqdm(
    csv_inventory["csv_path"],
    desc="Inspecting manifest CSV schemas",
):
    csv_path = Path(csv_path_str)

    try:
        sample = safe_read_csv(csv_path, nrows=20)

        schema_rows.append({
            "csv_path": str(csv_path),
            "file_name": csv_path.name,
            "rows_sampled": len(sample),
            "column_count": len(sample.columns),
            "columns": " | ".join(sample.columns.astype(str)),
            "sample_rows": sample.to_json(orient="records"),
        })

    except Exception as error:
        schema_rows.append({
            "csv_path": str(csv_path),
            "file_name": csv_path.name,
            "rows_sampled": np.nan,
            "column_count": np.nan,
            "columns": f"READ_ERROR: {type(error).__name__}: {error}",
            "sample_rows": "",
        })

schema_report = pd.DataFrame(schema_rows)

schema_report.to_csv(
    OUTPUTS["tables"] / "manifest_schema_inventory.csv",
    index=False,
)

display(schema_report[
    ["file_name", "column_count", "columns"]
])

Inspecting manifest CSV schemas:   0%|          | 0/35 [00:00<?, ?it/s]

,file_name,column_count,columns
0,class_distribution_summary.csv,9,crop | disease | label | total_images | train_...
1,crop_distribution_summary.csv,5,crop | images | classes | healthy_images | dis...
2,cross_split_exact_phash_matches.csv,18,image_path | relative_path | split | source_cl...
3,dataset_overview.csv,2,Metric | Value
4,eda_findings_and_modelling_implications.csv,4,Finding_ID | Observation | Quantitative_eviden...
5,excluded_exact_train_val_duplicates.csv,18,image_path | relative_path | split | source_cl...
6,healthy_diseased_summary.csv,5,crop | Diseased | Healthy | Total | Healthy (%)
7,image_metadata_summary.csv,9,split | crop | images | mean_width | std_width...
8,near_duplicate_candidates_phash_hamming_le4.csv,8,label | hamming_distance | train_image_path | ...
9,plantvillage_selected_manifest_clean.csv,18,image_path | relative_path | split | source_cl...


In [43]:
PATH_COLUMN_CANDIDATES = [
    "absolute_path",
    "abs_path",
    "image_path",
    "filepath",
    "file_path",
    "path",
    "relative_path",
    "rel_path",
    "relativepath",
    "filename",
    "file_name",
]

LABEL_COLUMN_CANDIDATES = [
    "class_name",
    "source_class",
    "label_name",
    "label",
    "class",
    "disease",
    "target",
]

SPLIT_COLUMN_CANDIDATES = [
    "final_split",
    "audited_split",
    "split",
    "historical_split",
    "dataset_split",
    "partition",
    "set",
]

HASH_COLUMN_CANDIDATES = [
    "sha256",
    "sha256_clean",
    "file_sha256",
    "hash",
    "phash",
    "perceptual_hash",
]

candidate_rows = []

for csv_path_str in tqdm(
    csv_inventory["csv_path"],
    desc="Scoring candidate manifests",
):
    csv_path = Path(csv_path_str)

    try:
        sample = safe_read_csv(csv_path, nrows=200)
        columns_lower = {
            column.lower(): column
            for column in sample.columns
        }

        matched_path = [
            columns_lower[column]
            for column in PATH_COLUMN_CANDIDATES
            if column in columns_lower
        ]

        matched_label = [
            columns_lower[column]
            for column in LABEL_COLUMN_CANDIDATES
            if column in columns_lower
        ]

        matched_split = [
            columns_lower[column]
            for column in SPLIT_COLUMN_CANDIDATES
            if column in columns_lower
        ]

        matched_hash = [
            columns_lower[column]
            for column in HASH_COLUMN_CANDIDATES
            if column in columns_lower
        ]

        score = (
            5 * bool(matched_path) +
            4 * bool(matched_label) +
            5 * bool(matched_split) +
            2 * bool(matched_hash)
        )

        split_values = ""

        if matched_split:
            values = (
                sample[matched_split[0]]
                .dropna()
                .astype(str)
                .str.lower()
                .unique()
                .tolist()
            )
            split_values = " | ".join(sorted(values)[:20])

            if any("calib" in value for value in values):
                score += 10

            if any("train" in value for value in values):
                score += 3

            if any(
                value in {"test", "internal_test", "valid", "validation"}
                for value in values
            ):
                score += 3

        candidate_rows.append({
            "csv_path": str(csv_path),
            "file_name": csv_path.name,
            "candidate_score": score,
            "path_columns": " | ".join(matched_path),
            "label_columns": " | ".join(matched_label),
            "split_columns": " | ".join(matched_split),
            "hash_columns": " | ".join(matched_hash),
            "sample_split_values": split_values,
        })

    except Exception as error:
        candidate_rows.append({
            "csv_path": str(csv_path),
            "file_name": csv_path.name,
            "candidate_score": -1,
            "path_columns": "",
            "label_columns": "",
            "split_columns": "",
            "hash_columns": "",
            "sample_split_values": f"ERROR: {error}",
        })

candidate_manifest_report = (
    pd.DataFrame(candidate_rows)
    .sort_values(
        ["candidate_score", "file_name"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

candidate_manifest_report.to_csv(
    OUTPUTS["tables"] / "candidate_manifest_ranking.csv",
    index=False,
)

display(candidate_manifest_report)

Scoring candidate manifests:   0%|          | 0/35 [00:00<?, ?it/s]

,csv_path,file_name,candidate_score,path_columns,label_columns,split_columns,hash_columns,sample_split_values
0,/kaggle/input/datasets/katakuricharlotte/crop-...,aux20_calibration.csv,26,image_path | relative_path | file_name,label | disease,final_split | split,sha256 | phash,calibration
1,/kaggle/input/datasets/katakuricharlotte/crop-...,main19_calibration.csv,26,image_path | relative_path | file_name,label | disease,final_split | split,sha256 | phash,calibration
2,/kaggle/input/datasets/katakuricharlotte/crop-...,aux20_internal_test.csv,19,image_path | relative_path | file_name,label | disease,final_split | split,sha256 | phash,internal_test
3,/kaggle/input/datasets/katakuricharlotte/crop-...,aux20_train.csv,19,image_path | relative_path | file_name,label | disease,final_split | split,sha256 | phash,train
4,/kaggle/input/datasets/katakuricharlotte/crop-...,cross_split_exact_phash_matches.csv,19,image_path | relative_path | file_name,label | disease,split,sha256 | phash,train | val
5,/kaggle/input/datasets/katakuricharlotte/crop-...,main19_internal_test.csv,19,image_path | relative_path | file_name,label | disease,final_split | split,sha256 | phash,internal_test
6,/kaggle/input/datasets/katakuricharlotte/crop-...,main19_train.csv,19,image_path | relative_path | file_name,label | disease,final_split | split,sha256 | phash,train
7,/kaggle/input/datasets/katakuricharlotte/crop-...,plantvillage_master_manifest_locked.csv,19,image_path | relative_path | file_name,label | disease,split,sha256 | phash,train
8,/kaggle/input/datasets/katakuricharlotte/crop-...,plantvillage_selected_manifest.csv,19,image_path | relative_path | file_name,label | disease,split,sha256 | phash,train
9,/kaggle/input/datasets/katakuricharlotte/crop-...,plantvillage_selected_manifest_clean.csv,19,image_path | relative_path | file_name,label | disease,split,sha256 | phash,train


In [44]:
# Frozen Main19 manifests only.
# Do NOT use aux20 files or plantvillage_master_manifest_locked.csv.

MANIFEST_DIR = Path(
    "/kaggle/input/datasets/katakuricharlotte/"
    "crop-manifestfiles/split_outputs/split_outputs/manifests/main19"
)

MAIN19_MANIFEST_PATHS = {
    "train": MANIFEST_DIR / "main19_train.csv",
    "calibration": MANIFEST_DIR / "main19_calibration.csv",
    "internal_test": MANIFEST_DIR / "main19_internal_test.csv",
}

print("Frozen Main19 manifest directory:")
print(MANIFEST_DIR)

for expected_split, csv_path in MAIN19_MANIFEST_PATHS.items():
    assert csv_path.exists(), (
        f"Missing {expected_split} manifest:\n{csv_path}"
    )

    size_mb = csv_path.stat().st_size / (1024 ** 2)

    print(
        f"{expected_split:14s} | "
        f"{csv_path.name:28s} | "
        f"{size_mb:8.3f} MB"
    )

Frozen Main19 manifest directory:
/kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/manifests/main19
train          | main19_train.csv             |    9.027 MB
calibration    | main19_calibration.csv       |    1.612 MB
internal_test  | main19_internal_test.csv     |    2.653 MB


In [45]:
manifest_frames = []

for expected_split, csv_path in MAIN19_MANIFEST_PATHS.items():
    frame = safe_read_csv(csv_path).copy()

    frame["manifest_file"] = csv_path.name
    frame["expected_frozen_split"] = expected_split

    manifest_frames.append(frame)

    print(
        f"{csv_path.name:30s} | "
        f"rows={len(frame):6d} | "
        f"columns={len(frame.columns):2d}"
    )

manifest_raw = pd.concat(
    manifest_frames,
    axis=0,
    ignore_index=True,
)

print("\nCombined frozen Main19 manifest shape:", manifest_raw.shape)

display(manifest_raw.head())

main19_train.csv               | rows= 18113 | columns=23
main19_calibration.csv         | rows=  3197 | columns=23
main19_internal_test.csv       | rows=  5322 | columns=23

Combined frozen Main19 manifest shape: (26632, 23)


,image_path,relative_path,split,source_class_folder,crop,disease,label,is_healthy,file_name,file_extension,...,sha256,phash,is_corrupt,exclude_from_evaluation,evaluation_split,exclude_near_duplicate_leakage,dataset_status,final_split,manifest_file,expected_frozen_split
0,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Corn_(maize)___Common_rust_/RS_Rust 1915...,train,Corn_(maize)___Common_rust_,corn,common_rust,corn__common_rust,0,RS_Rust 1915.JPG,.jpg,...,e12f62e6c710b002d2b4b9cbd39a30aeb3121da981efcb...,d0e21ec1b09fb596,0,0,train,0,retained,train,main19_train.csv,train
1,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Bacterial_spot/e41ca477-b751-45...,train,Tomato___Bacterial_spot,tomato,bacterial_spot,tomato__bacterial_spot,0,e41ca477-b751-45f7-9a19-3442349e45ef___GCREC_B...,.jpg,...,9d62d6860f47567f60ce3a636429c85c6ce0491c16a299...,ebc5c69f941a901b,0,0,train,0,retained,train,main19_train.csv,train
2,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Late_blight/370b9866-9a0d-4541-...,train,Tomato___Late_blight,tomato,late_blight,tomato__late_blight,0,370b9866-9a0d-4541-aaf1-cfefa1d2268c___RS_Late...,.jpg,...,a79cf89e81955ad69c8089b51540c5d98601cebb068bcb...,ffd0e20f983cb222,0,0,train,0,retained,train,main19_train.csv,train
3,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Bacterial_spot/509c3853-b405-44...,train,Tomato___Bacterial_spot,tomato,bacterial_spot,tomato__bacterial_spot,0,509c3853-b405-440f-bda3-59f1d7d238be___GCREC_B...,.jpg,...,e0a3a12ae7d66317124213d1d0276050aeab1a96d1b32b...,eba5c45a9c399178,0,0,train,0,retained,train,main19_train.csv,train
4,/kaggle/input/datasets/mohitsingh1804/plantvil...,train/Tomato___Bacterial_spot/71eeeb09-ce59-4b...,train,Tomato___Bacterial_spot,tomato,bacterial_spot,tomato__bacterial_spot,0,71eeeb09-ce59-4b58-80a4-a01c4348a218___GCREC_B...,.jpg,...,a277cddb7216ca7b58c08cd35568ad70e640fb60382cd8...,eee1840e9678cda9,0,0,train,0,retained,train,main19_train.csv,train


In [46]:
MANIFEST_PATH_COL = "image_path"
MANIFEST_RELATIVE_PATH_COL = "relative_path"
MANIFEST_LABEL_COL = "label"
MANIFEST_SPLIT_COL = "final_split"
MANIFEST_SHA_COL = "sha256"
MANIFEST_PHASH_COL = "phash"
MANIFEST_ID_COL = None

required_columns = [
    MANIFEST_PATH_COL,
    MANIFEST_RELATIVE_PATH_COL,
    MANIFEST_LABEL_COL,
    MANIFEST_SPLIT_COL,
    MANIFEST_SHA_COL,
    MANIFEST_PHASH_COL,
]

missing_columns = [
    column
    for column in required_columns
    if column not in manifest_raw.columns
]

assert not missing_columns, (
    f"Required columns missing: {missing_columns}\n"
    f"Available columns: {manifest_raw.columns.tolist()}"
)

print("Path column:", MANIFEST_PATH_COL)
print("Relative path column:", MANIFEST_RELATIVE_PATH_COL)
print("Label column:", MANIFEST_LABEL_COL)
print("Frozen split column:", MANIFEST_SPLIT_COL)
print("SHA-256 column:", MANIFEST_SHA_COL)
print("pHash column:", MANIFEST_PHASH_COL)

Path column: image_path
Relative path column: relative_path
Label column: label
Frozen split column: final_split
SHA-256 column: sha256
pHash column: phash


In [47]:
manifest = manifest_raw.copy()

manifest["class_name"] = (
    manifest[MANIFEST_LABEL_COL]
    .astype(str)
    .map(canonical_text)
)

# Use the original dedicated columns; do not try to reconstruct these
# from the normalized class_name.
manifest["crop"] = (
    manifest["crop"]
    .astype(str)
    .map(canonical_text)
)

manifest["condition"] = (
    manifest["disease"]
    .astype(str)
    .map(canonical_text)
)

manifest["split"] = (
    manifest[MANIFEST_SPLIT_COL]
    .astype(str)
    .map(normalise_split)
)

manifest["manifest_path"] = manifest[
    MANIFEST_PATH_COL
].astype(str)

manifest["relative_path"] = manifest[
    MANIFEST_RELATIVE_PATH_COL
].astype(str)

manifest["sha256_manifest"] = (
    manifest[MANIFEST_SHA_COL]
    .astype(str)
    .str.strip()
    .str.lower()
)

manifest["phash_manifest"] = (
    manifest[MANIFEST_PHASH_COL]
    .astype(str)
    .str.strip()
    .str.lower()
)

print("Total rows:", len(manifest))
print("Unique classes:", manifest["class_name"].nunique())

print("\nObserved final partitions:")
display(
    manifest["split"]
    .value_counts(dropna=False)
    .rename_axis("split")
    .reset_index(name="images")
)

display(
    manifest[
        [
            "class_name",
            "crop",
            "condition",
            "split",
            "manifest_file",
        ]
    ].head(15)
)

Total rows: 26632
Unique classes: 19

Observed final partitions:


,split,images
0,train,18113
1,internal_test,5322
2,calibration,3197


,class_name,crop,condition,split,manifest_file
0,corn_common_rust,corn,common_rust,train,main19_train.csv
1,tomato_bacterial_spot,tomato,bacterial_spot,train,main19_train.csv
2,tomato_late_blight,tomato,late_blight,train,main19_train.csv
3,tomato_bacterial_spot,tomato,bacterial_spot,train,main19_train.csv
4,tomato_bacterial_spot,tomato,bacterial_spot,train,main19_train.csv
5,tomato_septoria_leaf_spot,tomato,septoria_leaf_spot,train,main19_train.csv
6,tomato_septoria_leaf_spot,tomato,septoria_leaf_spot,train,main19_train.csv
7,corn_common_rust,corn,common_rust,train,main19_train.csv
8,tomato_tomato_yellow_leaf_curl_virus,tomato,tomato_yellow_leaf_curl_virus,train,main19_train.csv
9,tomato_septoria_leaf_spot,tomato,septoria_leaf_spot,train,main19_train.csv


In [48]:
file_split_check = (
    manifest
    .groupby(
        ["manifest_file", "expected_frozen_split"]
    )["split"]
    .agg(lambda values: sorted(values.dropna().unique()))
    .reset_index(name="observed_split_values")
)

display(file_split_check)

for _, row in file_split_check.iterrows():
    assert row["observed_split_values"] == [
        row["expected_frozen_split"]
    ], (
        f"Partition mismatch in {row['manifest_file']}:\n"
        f"Expected: {row['expected_frozen_split']}\n"
        f"Observed: {row['observed_split_values']}"
    )

print("PASS: Every CSV matches its declared frozen partition.")

,manifest_file,expected_frozen_split,observed_split_values
0,main19_calibration.csv,calibration,[calibration]
1,main19_internal_test.csv,internal_test,[internal_test]
2,main19_train.csv,train,[train]


PASS: Every CSV matches its declared frozen partition.


In [49]:
# These files should already exclude soybean.
# The filter remains as a safe validation guard and changes nothing on disk.

main19 = manifest.loc[
    ~manifest["crop"].str.contains("soy", na=False)
].copy()

main19 = main19.reset_index(drop=True)

print("Rows in Main19:", len(main19))
print("Unique Main19 classes:", main19["class_name"].nunique())
print("Crops:", sorted(main19["crop"].unique()))
print("Observed frozen splits:", sorted(main19["split"].unique()))

assert main19["class_name"].nunique() == 19, (
    f"Expected 19 classes; found "
    f"{main19['class_name'].nunique()}."
)

assert set(main19["split"].unique()) == {
    "train",
    "calibration",
    "internal_test",
}, (
    "Expected train, calibration, and internal_test. "
    f"Found: {sorted(main19['split'].unique())}"
)

assert set(main19["crop"].unique()) == {
    "bell_pepper",
    "corn",
    "potato",
    "tomato",
}, (
    "Unexpected crop inventory: "
    f"{sorted(main19['crop'].unique())}"
)

print("PASS: Complete frozen Main19 dataset is ready.")

Rows in Main19: 26632
Unique Main19 classes: 19
Crops: ['bell_pepper', 'corn', 'potato', 'tomato']
Observed frozen splits: ['calibration', 'internal_test', 'train']
PASS: Complete frozen Main19 dataset is ready.


In [50]:
split_order = [
    "train",
    "calibration",
    "internal_test",
]

split_counts = (
    main19["split"]
    .value_counts()
    .reindex(split_order)
    .rename("images")
    .reset_index()
    .rename(columns={"index": "split"})
)

display(split_counts)

FINAL_SPLIT_COUNTS = dict(
    zip(
        split_counts["split"],
        split_counts["images"].astype(int),
    )
)

FINAL_MAIN19_TOTAL = int(split_counts["images"].sum())

print("Authoritative frozen Main19 counts:")
print(FINAL_SPLIT_COUNTS)
print("Authoritative total:", FINAL_MAIN19_TOTAL)

,split,images
0,train,18113
1,calibration,3197
2,internal_test,5322


Authoritative frozen Main19 counts:
{'train': 18113, 'calibration': 3197, 'internal_test': 5322}
Authoritative total: 26632


In [51]:
observed_counts = (
    FINAL_SPLIT_COUNTS["train"],
    FINAL_SPLIT_COUNTS["calibration"],
    FINAL_SPLIT_COUNTS["internal_test"],
    FINAL_MAIN19_TOTAL,
)

print("Observed counts:", observed_counts)

known_manifest_versions = {
    (18113, 3194, 5322, 26629),
    (18113, 3197, 5322, 26632),
}

if observed_counts in known_manifest_versions:
    print("PASS: Matches a known frozen Main19 version.")
else:
    print(
        "WARNING: The frozen CSV counts differ from prior recorded "
        "versions. Treat these files as authoritative only after "
        "checking the displayed paths and source files."
    )

assert FINAL_SPLIT_COUNTS["train"] > 0
assert FINAL_SPLIT_COUNTS["calibration"] > 0
assert FINAL_SPLIT_COUNTS["internal_test"] > 0

Observed counts: (18113, 3197, 5322, 26632)
PASS: Matches a known frozen Main19 version.


In [52]:
class_split_counts = (
    main19
    .groupby(["class_name", "split"])
    .size()
    .rename("images")
    .reset_index()
)

class_split_pivot = (
    class_split_counts
    .pivot(
        index="class_name",
        columns="split",
        values="images",
    )
    .reindex(columns=split_order, fill_value=0)
    .fillna(0)
    .astype(int)
)

class_split_pivot["total"] = class_split_pivot.sum(axis=1)

display(class_split_pivot)

assert not (class_split_pivot[split_order] == 0).any().any(), (
    "At least one Main19 class has no images in a required partition."
)

class_split_pivot.to_csv(
    OUTPUTS["tables"] / "main19_class_by_split_counts.csv"
)

print("PASS: Every Main19 class occurs in every frozen partition.")

split,train,calibration,internal_test,total
class_name,,,,
bell_pepper_bacterial_spot,677,120,200,997
bell_pepper_healthy,1005,178,295,1478
corn_cercospora_leaf_spot_gray_leaf_spot,348,62,103,513
corn_common_rust,810,143,239,1192
corn_healthy,790,139,232,1161
corn_northern_leaf_blight,670,118,197,985
potato_early_blight,680,120,199,999
potato_healthy,103,18,31,152
potato_late_blight,680,120,200,1000


PASS: Every Main19 class occurs in every frozen partition.


In [53]:
reconciliation = pd.DataFrame({
    "split": split_order,
    "sum_of_class_counts": [
        int(class_split_pivot["train"].sum()),
        int(class_split_pivot["calibration"].sum()),
        int(class_split_pivot["internal_test"].sum()),
    ],
    "manifest_total": [
        FINAL_SPLIT_COUNTS["train"],
        FINAL_SPLIT_COUNTS["calibration"],
        FINAL_SPLIT_COUNTS["internal_test"],
    ],
})

reconciliation["difference"] = (
    reconciliation["sum_of_class_counts"] -
    reconciliation["manifest_total"]
)

display(reconciliation)

assert (reconciliation["difference"] == 0).all(), (
    "Class totals do not reconcile to manifest totals."
)

print("PASS: Split totals reconcile exactly.")

,split,sum_of_class_counts,manifest_total,difference
0,train,18113,18113,0
1,calibration,3197,3197,0
2,internal_test,5322,5322,0


PASS: Split totals reconcile exactly.


In [ ]:
def count_images_in_directory(path, max_files=None):
    count = 0

    for child in path.rglob("*"):
        if child.is_file() and child.suffix.lower() in IMAGE_EXTENSIONS:
            count += 1

            if max_files is not None and count >= max_files:
                break

    return count

candidate_image_dirs = []

for path in PLANTVILLAGE_ROOT.rglob("*"):
    if path.is_dir():
        try:
            image_count_preview = count_images_in_directory(
                path,
                max_files=1000,
            )

            if image_count_preview > 0:
                candidate_image_dirs.append({
                    "directory": str(path),
                    "relative_directory": str(
                        path.relative_to(PLANTVILLAGE_ROOT)
                    ),
                    "preview_image_count_capped": image_count_preview,
                })

        except PermissionError:
            pass

candidate_image_dirs_df = (
    pd.DataFrame(candidate_image_dirs)
    .sort_values(
        "preview_image_count_capped",
        ascending=False,
    )
    .head(30)
)

display(candidate_image_dirs_df)

# In most Kaggle PlantVillage datasets this resolves automatically.
PLANTVILLAGE_IMAGE_ROOT = PLANTVILLAGE_ROOT

print("Image root currently set to:", PLANTVILLAGE_IMAGE_ROOT)

In [ ]:
filename_index = defaultdict(list)

for image_path in tqdm(
    PLANTVILLAGE_IMAGE_ROOT.rglob("*"),
    desc="Indexing PlantVillage image files",
):
    if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
        filename_index[image_path.name].append(image_path)

total_indexed_images = sum(
    len(paths)
    for paths in filename_index.values()
)

print("Unique image filenames:", len(filename_index))
print("Total image files indexed:", total_indexed_images)

In [ ]:
def resolve_manifest_path(raw_path):
    raw_path = str(raw_path).strip()
    candidate = Path(raw_path)

    # Case 1: absolute manifest path exists in the current runtime.
    if candidate.is_absolute() and candidate.exists():
        return candidate

    # Case 2: direct relative path from detected image root.
    direct = PLANTVILLAGE_IMAGE_ROOT / candidate
    if direct.exists():
        return direct

    # Case 3: remove leading separator and try again.
    direct = PLANTVILLAGE_IMAGE_ROOT / raw_path.lstrip("/\\")
    if direct.exists():
        return direct

    # Case 4: find by basename, then disambiguate using suffix components.
    candidates = filename_index.get(candidate.name, [])

    if len(candidates) == 1:
        return candidates[0]

    if len(candidates) > 1:
        raw_parts = [
            part.lower()
            for part in candidate.parts
        ]

        for suffix_length in [5, 4, 3, 2]:
            if len(raw_parts) < suffix_length:
                continue

            wanted_suffix = tuple(raw_parts[-suffix_length:])

            matches = []

            for candidate_path in candidates:
                candidate_parts = tuple(
                    part.lower()
                    for part in candidate_path.parts[-suffix_length:]
                )

                if candidate_parts == wanted_suffix:
                    matches.append(candidate_path)

            if len(matches) == 1:
                return matches[0]

    return None

resolved_paths = []

for raw_path in tqdm(
    main19["manifest_path"],
    desc="Resolving manifest image paths",
):
    result = resolve_manifest_path(raw_path)
    resolved_paths.append(
        str(result) if result is not None else np.nan
    )

main19["resolved_path"] = resolved_paths

unresolved = main19[
    main19["resolved_path"].isna()
].copy()

print("Resolved:", int(main19["resolved_path"].notna().sum()))
print("Unresolved:", len(unresolved))

unresolved.to_csv(
    OUTPUTS["tables"] / "unresolved_manifest_paths.csv",
    index=False,
)

display(unresolved.head(20))

In [ ]:
assert main19["resolved_path"].notna().all(), (
    "Some manifest rows could not be resolved to PlantVillage image files. "
    "Do not proceed. Inspect unresolved_manifest_paths.csv and correct "
    "PLANTVILLAGE_IMAGE_ROOT or the chosen manifest."
)

assert main19["resolved_path"].map(
    lambda path: Path(path).exists()
).all(), "At least one resolved path does not exist."

print("PASS: Every Main19 manifest record resolves to an existing image.")

In [ ]:
SAMPLE_PER_SPLIT = 500

sample_rows = []

for split_name, split_df in main19.groupby("split"):
    n = min(SAMPLE_PER_SPLIT, len(split_df))

    split_sample = split_df.sample(
        n=n,
        random_state=SEED,
    )

    for _, row in tqdm(
        split_sample.iterrows(),
        total=len(split_sample),
        desc=f"Verifying {split_name}",
    ):
        path = Path(row["resolved_path"])

        try:
            with Image.open(path) as image:
                image.verify()

            valid = True
            error_message = ""

        except Exception as error:
            valid = False
            error_message = f"{type(error).__name__}: {error}"

        sample_rows.append({
            "split": split_name,
            "class_name": row["class_name"],
            "resolved_path": str(path),
            "is_readable": valid,
            "error": error_message,
        })

readability_sample = pd.DataFrame(sample_rows)

readability_summary = (
    readability_sample.groupby("split")["is_readable"]
    .agg(["count", "sum", "mean"])
    .rename(columns={
        "count": "sampled_images",
        "sum": "readable_images",
        "mean": "readable_fraction",
    })
    .reset_index()
)

readability_sample.to_csv(
    OUTPUTS["tables"] / "path_readability_sample.csv",
    index=False,
)

readability_summary.to_csv(
    OUTPUTS["tables"] / "path_readability_summary.csv",
    index=False,
)

display(readability_summary)

assert readability_sample["is_readable"].all(), (
    "One or more sampled images could not be opened. "
    "Inspect path_readability_sample.csv."
)

In [ ]:
id_integrity = {
    "rows": len(main19),
    "manifest_id_column_present": MANIFEST_ID_COL is not None,
    "sha256_column_present": MANIFEST_SHA_COL is not None,
    "phash_column_present": MANIFEST_PHASH_COL is not None,
}

if MANIFEST_ID_COL is not None:
    non_empty_ids = main19["image_id_manifest"].replace(
        {"nan": np.nan, "": np.nan}
    ).dropna()

    id_integrity["non_empty_manifest_ids"] = len(non_empty_ids)
    id_integrity["duplicate_manifest_ids"] = int(
        non_empty_ids.duplicated().sum()
    )
else:
    id_integrity["non_empty_manifest_ids"] = 0
    id_integrity["duplicate_manifest_ids"] = np.nan

if MANIFEST_SHA_COL is not None:
    non_empty_hashes = main19["sha256_manifest"].replace(
        {"nan": np.nan, "": np.nan}
    ).dropna()

    id_integrity["non_empty_sha256"] = len(non_empty_hashes)
    id_integrity["duplicate_sha256_values"] = int(
        non_empty_hashes.duplicated().sum()
    )
else:
    id_integrity["non_empty_sha256"] = 0
    id_integrity["duplicate_sha256_values"] = np.nan

id_integrity_df = pd.DataFrame([id_integrity])

id_integrity_df.to_csv(
    OUTPUTS["tables"] / "manifest_identifier_integrity.csv",
    index=False,
)

display(id_integrity_df)

In [ ]:
if MANIFEST_SHA_COL is None:
    print(
        "No SHA-256 column was found in the selected manifest. "
        "Skip this cell and use Cell 27 to compute hashes for a sample."
    )

else:
    hash_table = main19[
        ["split", "class_name", "manifest_path",
         "resolved_path", "sha256_manifest"]
    ].copy()

    hash_table = hash_table.loc[
        hash_table["sha256_manifest"].notna() &
        (hash_table["sha256_manifest"] != "nan") &
        (hash_table["sha256_manifest"] != "")
    ].copy()

    sha_split_count = (
        hash_table.groupby("sha256_manifest")["split"]
        .nunique()
    )

    cross_split_sha = sha_split_count[
        sha_split_count > 1
    ].index

    exact_cross_split_duplicates = (
        hash_table[
            hash_table["sha256_manifest"].isin(cross_split_sha)
        ]
        .sort_values(["sha256_manifest", "split"])
        .reset_index(drop=True)
    )

    exact_duplicate_summary = pd.DataFrame([{
        "images_with_manifest_sha256": len(hash_table),
        "unique_sha256_values": hash_table["sha256_manifest"].nunique(),
        "duplicate_sha256_values_anywhere": int(
            hash_table["sha256_manifest"].duplicated().sum()
        ),
        "cross_split_duplicate_hashes": len(cross_split_sha),
        "cross_split_duplicate_rows": len(
            exact_cross_split_duplicates
        ),
    }])

    exact_cross_split_duplicates.to_csv(
        OUTPUTS["tables"] / "exact_sha256_cross_split_duplicates.csv",
        index=False,
    )

    exact_duplicate_summary.to_csv(
        OUTPUTS["tables"] / "exact_sha256_duplicate_summary.csv",
        index=False,
    )

    display(exact_duplicate_summary)
    display(exact_cross_split_duplicates.head(30))

In [ ]:
HASH_VALIDATION_SAMPLE_SIZE = 100

hash_validation_rows = []

for split_name, split_df in main19.groupby("split"):
    n = min(
        HASH_VALIDATION_SAMPLE_SIZE,
        len(split_df),
    )

    subset = split_df.sample(
        n=n,
        random_state=SEED,
    )

    for _, row in tqdm(
        subset.iterrows(),
        total=len(subset),
        desc=f"Hash validation: {split_name}",
    ):
        computed_hash = sha256_file(Path(row["resolved_path"]))

        manifest_hash = row["sha256_manifest"]

        if MANIFEST_SHA_COL is not None:
            match = (
                str(computed_hash).lower() ==
                str(manifest_hash).strip().lower()
            )
        else:
            match = np.nan

        hash_validation_rows.append({
            "split": split_name,
            "resolved_path": row["resolved_path"],
            "computed_sha256": computed_hash,
            "manifest_sha256": manifest_hash,
            "matches_manifest_hash": match,
        })

hash_validation = pd.DataFrame(hash_validation_rows)

hash_validation.to_csv(
    OUTPUTS["tables"] / "sampled_sha256_validation.csv",
    index=False,
)

if MANIFEST_SHA_COL is not None:
    hash_validation_summary = (
        hash_validation.groupby("split")[
            "matches_manifest_hash"
        ]
        .agg(["count", "sum", "mean"])
        .reset_index()
        .rename(columns={
            "count": "sampled",
            "sum": "matching_hashes",
            "mean": "match_fraction",
        })
    )

    display(hash_validation_summary)

    assert hash_validation["matches_manifest_hash"].all(), (
        "At least one sampled SHA-256 value disagrees with its manifest. "
        "This indicates the manifest may reference a different dataset version."
    )

    print("PASS: Sampled hashes match manifest SHA-256 values.")
else:
    print(
        "Manifest has no SHA-256 column; computed hashes were still exported."
    )

In [ ]:
METADATA_SAMPLE_PER_SPLIT = 300

metadata_rows = []

for split_name, split_df in main19.groupby("split"):
    n = min(
        METADATA_SAMPLE_PER_SPLIT,
        len(split_df),
    )

    subset = split_df.sample(
        n=n,
        random_state=SEED,
    )

    for _, row in tqdm(
        subset.iterrows(),
        total=len(subset),
        desc=f"Image metadata: {split_name}",
    ):
        path = Path(row["resolved_path"])

        try:
            with Image.open(path) as image:
                width, height = image.size
                mode = image.mode
                image_format = image.format

            metadata_rows.append({
                "split": split_name,
                "class_name": row["class_name"],
                "path": str(path),
                "width": width,
                "height": height,
                "aspect_ratio": width / height,
                "mode": mode,
                "format": image_format,
                "file_size_bytes": path.stat().st_size,
                "readable": True,
                "error": "",
            })

        except Exception as error:
            metadata_rows.append({
                "split": split_name,
                "class_name": row["class_name"],
                "path": str(path),
                "width": np.nan,
                "height": np.nan,
                "aspect_ratio": np.nan,
                "mode": "",
                "format": "",
                "file_size_bytes": np.nan,
                "readable": False,
                "error": f"{type(error).__name__}: {error}",
            })

image_metadata_sample = pd.DataFrame(metadata_rows)

image_metadata_sample.to_csv(
    OUTPUTS["tables"] / "image_metadata_sample.csv",
    index=False,
)

metadata_summary = (
    image_metadata_sample.groupby("split")
    .agg(
        sampled_images=("path", "count"),
        readable_fraction=("readable", "mean"),
        median_width=("width", "median"),
        median_height=("height", "median"),
        median_file_size_bytes=("file_size_bytes", "median"),
    )
    .reset_index()
)

display(metadata_summary)

In [ ]:
plot_table = class_split_pivot[
    ["train", "calibration", "internal_test"]
].copy()

plot_table = plot_table.sort_values(
    "train",
    ascending=True,
)

fig, ax = plt.subplots(figsize=(14, 10))

left = np.zeros(len(plot_table))

colors = {
    "train": "#4C78A8",
    "calibration": "#F2CF5B",
    "internal_test": "#E45756",
}

for split_name in [
    "train",
    "calibration",
    "internal_test",
]:
    values = plot_table[split_name].to_numpy()

    ax.barh(
        plot_table.index,
        values,
        left=left,
        label=split_name,
        color=colors[split_name],
    )

    left += values

ax.set_title("Main19 Frozen Manifest: Class Distribution by Split")
ax.set_xlabel("Number of images")
ax.set_ylabel("Class")
ax.legend(title="Split")
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()

plot_path = OUTPUTS["figures"] / "main19_class_distribution.png"

plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", plot_path)

In [ ]:
plot_counts = pd.DataFrame({
    "split": [
        "train",
        "calibration",
        "internal_test",
    ],
    "images": [
        EXPECTED_FINAL_COUNTS["train"],
        EXPECTED_FINAL_COUNTS["calibration"],
        EXPECTED_FINAL_COUNTS["internal_test"],
    ],
})

fig, ax = plt.subplots(figsize=(9, 6))

bars = ax.bar(
    plot_counts["split"],
    plot_counts["images"],
    color=["#4C78A8", "#F2CF5B", "#E45756"],
)

for bar, value in zip(bars, plot_counts["images"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        value,
        f"{value:,}",
        ha="center",
        va="bottom",
        fontsize=11,
    )

ax.set_title("Final Audited Main19 Partition Counts")
ax.set_ylabel("Images")
ax.grid(axis="y", alpha=0.25)

plt.tight_layout()

plot_path = OUTPUTS["figures"] / "main19_final_split_counts.png"

plt.savefig(
    plot_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

print("Saved:", plot_path)

In [ ]:
manifest_export = main19[
    [
        "manifest_path",
        "resolved_path",
        "class_name",
        "split",
        "sha256_manifest",
        "phash_manifest",
        "image_id_manifest",
    ]
].copy()

manifest_export = manifest_export.sort_values(
    ["split", "class_name", "resolved_path"]
).reset_index(drop=True)

frozen_export_path = (
    OUTPUTS["reports"] / "main19_resolved_frozen_manifest.csv"
)

manifest_export.to_csv(
    frozen_export_path,
    index=False,
)

manifest_fingerprint = sha256_file(frozen_export_path)

run_metadata = {
    "created_utc": utc_now(),
    "selected_manifest": str(MANIFEST_PATH),
    "plantvillage_root": str(PLANTVILLAGE_ROOT),
    "plantvillage_image_root": str(PLANTVILLAGE_IMAGE_ROOT),
    "rows_main19": int(len(main19)),
    "main19_classes": int(main19["class_name"].nunique()),
    "split_counts": {
        key: int(value)
        for key, value in EXPECTED_FINAL_COUNTS.items()
    },
    "resolved_manifest_sha256": manifest_fingerprint,
    "path_column": MANIFEST_PATH_COL,
    "label_column": MANIFEST_LABEL_COL,
    "split_column": MANIFEST_SPLIT_COL,
    "sha256_column": MANIFEST_SHA_COL,
    "phash_column": MANIFEST_PHASH_COL,
    "image_id_column": MANIFEST_ID_COL,
}

metadata_path = (
    OUTPUTS["reports"] / "main19_dataset_audit_metadata.json"
)

with open(metadata_path, "w") as file:
    json.dump(run_metadata, file, indent=2)

print(json.dumps(run_metadata, indent=2))

In [ ]:
final_report = {
    "selected_manifest": str(MANIFEST_PATH),
    "main19_images": int(len(main19)),
    "main19_classes": int(main19["class_name"].nunique()),
    "train_images": int(
        (main19["split"] == "train").sum()
    ),
    "calibration_images": int(
        (main19["split"] == "calibration").sum()
    ),
    "internal_test_images": int(
        (main19["split"] == "internal_test").sum()
    ),
    "all_paths_resolved": bool(
        main19["resolved_path"].notna().all()
    ),
    "all_sampled_images_readable": bool(
        readability_sample["is_readable"].all()
    ),
    "all_classes_have_train_support": bool(
        (class_split_pivot["train"] > 0).all()
    ),
    "all_classes_have_calibration_support": bool(
        (class_split_pivot["calibration"] > 0).all()
    ),
    "all_classes_have_test_support": bool(
        (class_split_pivot["internal_test"] > 0).all()
    ),
    "resolved_manifest_fingerprint": manifest_fingerprint,
}

if MANIFEST_SHA_COL is not None:
    final_report[
        "exact_cross_split_duplicate_hashes"
    ] = int(len(cross_split_sha))

    final_report[
        "exact_cross_split_duplicate_rows"
    ] = int(len(exact_cross_split_duplicates))

final_report_df = pd.DataFrame([
    final_report
])

final_report_df.to_csv(
    OUTPUTS["reports"] / "main19_final_audit_report.csv",
    index=False,
)

display(final_report_df.T)

print("\nAudit outputs are available in:")
print(OUTPUTS["root"])

In [ ]:
handoff = {
    "manifest_path": str(MANIFEST_PATH),
    "plantvillage_image_root": str(PLANTVILLAGE_IMAGE_ROOT),
    "path_column": MANIFEST_PATH_COL,
    "label_column": MANIFEST_LABEL_COL,
    "split_column": MANIFEST_SPLIT_COL,
    "sha256_column": MANIFEST_SHA_COL,
    "phash_column": MANIFEST_PHASH_COL,
    "image_id_column": MANIFEST_ID_COL,
    "main19_classes": main19_class_list,
    "expected_counts": EXPECTED_FINAL_COUNTS,
    "resolved_manifest_path": str(frozen_export_path),
    "resolved_manifest_fingerprint": manifest_fingerprint,
}

handoff_path = (
    OUTPUTS["reports"] / "training_notebook_handoff.json"
)

with open(handoff_path, "w") as file:
    json.dump(handoff, file, indent=2)

print("Training-notebook handoff saved to:")
print(handoff_path)

print("\nUse this handoff file and main19_resolved_frozen_manifest.csv")
print("for the final model-training notebook.")